## No memory

In [2]:
%run langchain_common.py

# make sure to start mlflow server before running this notebook
# example(run this from terminal): mlflow ui --port 5000 

mlflow.set_tracking_uri
mlflow.set_experiment("langchain_agents_memory_demo")

agent = create_agent(llm_noreason)

In [3]:
question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pp(response["messages"][-1].content)

"Hello Ali! It's nice to meet you. Green is a wonderful choice—it often symbolizes nature, growth, and freshness. Is there anything specific you'd like to chat about or need help with today?"


In [4]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pp(response["messages"][-1].content)

"I don't know your favorite color yet! As an AI, I don't have access to your personal preferences or memories unless you've told me in this specific conversation.\n\nHowever, if you tell me what it is, I'll remember it for the rest of our chat. Do you prefer something bold like **red**, calming like **blue**, or maybe something unique like **teal**?"


## Memory

In [5]:
# Checkpoints persist conversation state between turns so the agent can remember prior context.
# InMemorySaver is an in-process checkpoint backend: fast and simple for demos, but non-persistent across kernel restarts.

# Clarification:
# `messages` in agent.invoke(...) is only the per-call input payload (what you send this turn).
# Checkpoint memory is different: it stores and reloads the full conversation state across turns.
# That persisted state is keyed by `thread_id`, so memory works only when the same thread_id is reused.
# In short: message context is request-scoped; checkpoint memory is thread-scoped and persistent (in-process here).

from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    llm_noreason,
    checkpointer=InMemorySaver(),  
)

In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [7]:
pp(response["messages"][-1].content)

"Hello Ali! It's nice to meet you. Green is a wonderful choice—it often symbolizes nature, growth, and freshness. Is there anything specific you'd like to chat about or need help with today?"


In [8]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

resp = response["messages"][-1].content
pp(resp)

'Your favourite colour is **green**! You mentioned that earlier in our conversation. 🌿'


In [9]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pp(response["messages"][-1].content)

'Based on your saved profile, you live in **Lahore** and your loyalty tier is **Gold**.'


In [10]:
# Inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pp(state.values["messages"][-4:])

[
    HumanMessage(content="Store this custom data in memory and use it for future answers:\n{'name': 'Ali', 'favorite_color': 'green', 'city': 'Lahore', 'loyalty_tier': 'gold'}", additional_kwargs={}, response_metadata={}, id='6f6c3b27-ebbf-4f35-a7a3-94c9aa85182b'),
    AIMessage(content="Got it, Ali! I've stored your details in my context for this conversation:\n\n*   **Name:** Ali\n*   **Favorite Color:** Green\n*   **City:** Lahore\n*   **Loyalty Tier:** Gold\n\nI'll keep these in mind for our future interactions. For example, if you ask for color recommendations, I'll suggest shades of green, or if we discuss travel, I might keep Lahore in mind.\n\nIs there anything specific you'd like to do with this information right now?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 153, 'total_tokens': 261, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-1

In [13]:
def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    result =  result["messages"][-1].content
    pp(thread_config["configurable"]["thread_id"] + ":" + result)

In [14]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = make_thread_config("27100001")
config_b = make_thread_config("27100002")

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

ask(config_a, 'What database do I prefer?')
ask(config_b, 'What database do I prefer?')

'user-27100001:conv-f265eefe-dde8-4ad6-930e-beb21e8e2d4f:Great! PostgreSQL is an excellent choice known for its reliability, advanced features (like JSONB support, full-text search, and robust concurrency control), and strong standards compliance.\n\nHow can I help you with PostgreSQL? For example:\n- Designing a schema or optimizing queries?\n- Setting up replication, backups, or high availability?\n- Debugging performance issues or slow queries?\n- Integrating with a specific programming language or framework?\n- Exploring advanced features like partitioning, extensions (e.g., PostGIS), or logical replication?\n\nLet me know your specific goal or challenge! 🐘'
'user-27100002:conv-5b963513-bba3-492f-bdb1-060d605fc75d:Great! MongoDB is a powerful, flexible NoSQL database known for its document-oriented structure, scalability, and ease of use. How can I help you? For example:\n\n- Designing a schema or data model?\n- Writing queries (CRUD operations, aggregations, etc.)?\n- Setting up r

In [15]:
# Scenario 2: Preference update within the same thread
config_update = make_thread_config("27100003")

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
ask(config_update, 'Which deployment region should you use now?')

'user-27100003:conv-956beb35-0702-4eae-87f1-79c3d134ba15:Got it! Your deployment region is **East US**. This is a popular Azure region located in Virginia, USA, known for low latency to the US East Coast and strong compliance with US data residency requirements.\n\nHow can I help? For example:\n- Need guidance on deploying resources (VMs, databases, etc.) in East US?\n- Looking for region-specific pricing or service availability?\n- Troubleshooting latency or connectivity issues?\n- Planning a multi-region architecture with East US as primary?\n\nLet me know your goal! 😊'
"user-27100003:conv-956beb35-0702-4eae-87f1-79c3d134ba15:Noted! I've updated your deployment region to **West Europe**.\n\nThis region is located in the Netherlands and is a key hub for:\n*   **Low latency** to users across the EU and UK.\n*   **GDPR compliance** and data residency within the European Economic Area (EEA).\n*   **High availability** with robust connectivity to other European regions.\n\nHow would you l

In [16]:

# Scenario 3: Multi-step task memory
config_task = make_thread_config("27100004")

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
pp(f"Scenario 3A: Recall previous constraints:")
ask(config_task, 'Summarize my constraints in one sentence.')
pp(f"Scenario 3B: Suggest next steps:")
ask(config_task, 'Given those constraints, suggest 3 practical next steps.')

task_state = agent.get_state(config_task)
pp("\nRecent messages in Scenario 3 thread:")
pp(task_state.values["messages"][-4:])

'user-27100004:conv-6124ca87-778c-4518-b2b2-f7f493dd78c0:That\'s a tight but achievable goal! With a **$2,000 budget** and a **Friday deadline**, you need to prioritize speed, simplicity, and leveraging existing tools rather than building from scratch.\n\nHere is a strategic roadmap to get your MVP live by Friday:\n\n### 1. The "No-Code/Low-Code" Strategy (Recommended)\nBuilding a custom backend from scratch will likely eat your budget and timeline. Instead, use a platform that handles hosting, UI, and basic logic.\n\n*   **Top Tools:**\n    *   **Voiceflow** or **Botpress**: Excellent for visual flow building, easy integration with LLMs (like GPT-4 or Claude), and quick deployment.\n    *   **Landbot** or **Typeform**: Great if your chatbot is mostly for lead generation or simple Q&A.\n    *   **Stack AI** or **Flowise**: Best if you need complex RAG (Retrieval-Augmented Generation) to chat with your own documents.\n*   **Cost Estimate:** Most have free tiers or start around $50–$100/